# Experiment 2 — Checkpoint Trajectory Analysis (fully on Colab)

**Question:** does retrieval performance peak *earlier* in training than triplet
validation accuracy — i.e. was Experiment 1's model-selection criterion wrong?

This notebook runs **end to end on Colab with no local steps and no uploads.**
It clones the repo (which contains the raw dataset), regenerates the splits and
triplets, trains identically to Experiment 1 but keeps a checkpoint every 50
steps, evaluates every checkpoint on the **validation** queries, selects the best
by validation MRR@10, evaluates that single checkpoint once on **test**, and
packages all results for download.

**Only edit `REPO_URL` below, set Runtime → GPU, then Runtime → Run all.**

Nothing here changes the methodology: dataset, split, triplet logic, loss,
optimizer, LR, batch size, epochs, margin, seed, retrieval algorithm, and
metrics are all unchanged. The only differences from Experiment 1 are checkpoint
retention (`--save-steps 50 --keep-all-checkpoints`) and post-hoc evaluation.

In [ ]:
# --- The ONLY cell you edit ---
REPO_URL = "https://github.com/YOUR_USERNAME/distractor.git"  # <-- set this

In [ ]:
# --- Verify GPU (Runtime > Change runtime type > GPU) ---
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > GPU"
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
# --- Clone repo (includes datasets/) and install pinned deps ---
import os
if not os.path.exists("distractor"):
    !git clone {REPO_URL} distractor
%cd distractor
%pip install -q -r requirements_gpu.txt
# Confirm the raw competition CSVs came with the clone (no upload needed)
assert os.path.exists("datasets/train.csv"), "datasets/train.csv missing from repo"
print("Raw dataset present:", os.listdir("datasets"))

In [ ]:
# --- Auto-generate splits if missing (Stage 01) ---
import os
if not os.path.exists("outputs/results/train_qdp.csv"):
    !python scripts/01_prepare_dataset.py
else:
    print("Splits already present — skipping Stage 01.")

In [ ]:
# --- Auto-generate triplets if missing (Stage 03) ---
import os
if not os.path.exists("outputs/triplets/train_triplets.jsonl"):
    !python scripts/03_create_triplets.py
else:
    print("Triplets already present — skipping Stage 03.")

In [ ]:
# --- Train with full checkpoint retention (identical to Exp 1 otherwise) ---
!python scripts/train_gpu.py --save-steps 50 --eval-steps 50 --keep-all-checkpoints

In [ ]:
# --- Sanity check: all checkpoints retained ---
import os
ckpts = sorted(os.listdir("outputs/models/finetuned_pedagogical/checkpoints"))
print(f"{len([c for c in ckpts if c.startswith('checkpoint-')])} checkpoints kept:")
print([c for c in ckpts if c.startswith('checkpoint-')])

In [ ]:
# --- Evaluate every checkpoint (validation-first; test touched once) ---
!python scripts/06_evaluate_checkpoints.py

In [ ]:
# --- Show the auto-generated report inline ---
print(open("outputs/results/checkpoint_trajectory_report.md").read())

In [ ]:
# --- Package all outputs and download ---
!zip -qr exp2_results.zip \
  outputs/results/checkpoint_metrics_val.csv \
  outputs/results/final_test_results.csv \
  outputs/results/checkpoint_trajectory_report.md \
  outputs/figures/exp2_trajectory \
  outputs/models/finetuned_pedagogical/training_log.json
from google.colab import files
files.download("exp2_results.zip")